# Australian Fuel Market : Competitor Pricing Analysis
## Notebook 1: Data Acquisition and Cleaning

**Source:** FuelWatch Western Australia — Department of Energy, Mines, Industry Regulation and Safety  
**Licence:** Creative Commons Attribution 4.0  
**Data coverage:** Daily station-level retail fuel prices across Western Australia since January 2001

---

### Why FuelWatch?

FuelWatch is one of the most granular public fuel price datasets in the world. Unlike the ACCC's aggregate weekly reports, which give city-level averages; FuelWatch records the price set by every individual station, every day, broken down by brand and fuel type. This means we can directly compare 7-Eleven's pricing behaviour against specific competitors at the brand level over time.

Western Australia's FuelWatch scheme requires all fuel retailers to lock in a single price for the day at 6am and publish it to the government database. This 24-hour price lock creates an unusually clean dataset, there are no intra-day fluctuations, no missing timestamps, and the brand attribution is reliable.

---

### Step 0 — Download the raw data (manual, do this once)

Before running this notebook, you need to download the monthly CSV files from FuelWatch.

**Instructions:**
1. Go to: https://www.fuelwatch.wa.gov.au/retail/historic
2. For each month from **January 2022 to April 2025**, select the year and month from the dropdowns and click Download
3. Save each file into a folder called `data/raw/` in the same directory as this notebook
4. The files will be named something like `FuelWatchRetail_2024_01.csv` — keep the original names

That is 40 files in total. It takes around 10 minutes to download them all.

**Folder structure you should have before running:**
```
fuel-market-competitor-pricing-australia/
├── data/
│   ├── raw/          ← put all 40 downloaded CSV files here
│   └── processed/    ← this notebook will create files here
├── notebook1_data_acquisition.ipynb
├── notebook2_competitor_analysis.ipynb
└── notebook3_weekly_reporting_pipeline.ipynb
```

In [6]:
# Install pyarrow if you don't already have it (needed to write .parquet files)
# Run this cell once, then you won't need it again
# !pip install pyarrow

In [7]:
import pandas as pd
import numpy as np
import glob
import os
from pathlib import Path

print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

pandas version: 2.2.2
numpy version: 1.26.4


In [8]:
# Create the processed data folder if it doesn't exist yet
Path('data/processed').mkdir(parents=True, exist_ok=True)

# Find all CSV files in the raw folder
raw_files = sorted(glob.glob('data/raw/*.csv'))

print(f"Found {len(raw_files)} CSV files in data/raw/")
print("\nFirst few files:")
for f in raw_files[:5]:
    print(f"  {f}")

Found 41 CSV files in data/raw/

First few files:
  data/raw/FuelWatchRetail-01-2022.csv
  data/raw/FuelWatchRetail-01-2023.csv
  data/raw/FuelWatchRetail-01-2024.csv
  data/raw/FuelWatchRetail-01-2025.csv
  data/raw/FuelWatchRetail-02-2022.csv


### Step 1 — Inspect a single file

Before loading everything, we look at one file to understand the column names, data types, and any quirks.

In [10]:
# Load just the first file to inspect its structure
sample = pd.read_csv(raw_files[0])

print("Shape:", sample.shape)
print("\nColumn names:")
print(sample.columns.tolist())
print("\nFirst 5 rows:")
sample.head()

Shape: (82108, 11)

Column names:
['PUBLISH_DATE', 'TRADING_NAME', 'BRAND_DESCRIPTION', 'PRODUCT_DESCRIPTION', 'PRODUCT_PRICE', 'ADDRESS', 'LOCATION', 'POSTCODE', 'AREA_DESCRIPTION', 'REGION_DESCRIPTION', 'Unnamed: 10']

First 5 rows:


,PUBLISH_DATE,TRADING_NAME,BRAND_DESCRIPTION,PRODUCT_DESCRIPTION,PRODUCT_PRICE,ADDRESS,LOCATION,POSTCODE,AREA_DESCRIPTION,REGION_DESCRIPTION,Unnamed: 10
0,01/01/2022,53 Mile Roadhouse,United,ULP,153.9,31 South Western Hwy,PINJARRA,6208,Murray,Peel,NaN
1,01/01/2022,53 Mile Roadhouse,United,Diesel,150.9,31 South Western Hwy,PINJARRA,6208,Murray,Peel,NaN
2,01/01/2022,53 Mile Roadhouse,United,98 RON,169.9,31 South Western Hwy,PINJARRA,6208,Murray,Peel,NaN
3,01/01/2022,7-Eleven Ascot,7-Eleven,ULP,174.9,194 Great Eastern Hwy,ASCOT,6104,South of River,Metro,NaN
4,01/01/2022,7-Eleven Ascot,7-Eleven,Brand Diesel,155.7,194 Great Eastern Hwy,ASCOT,6104,South of River,Metro,NaN


In [11]:
# Check data types and null counts in the sample file
print("Data types and null counts:")
print(sample.dtypes)
print("\nNull values per column:")
print(sample.isnull().sum())

Data types and null counts:
PUBLISH_DATE            object
TRADING_NAME            object
BRAND_DESCRIPTION       object
PRODUCT_DESCRIPTION     object
PRODUCT_PRICE          float64
ADDRESS                 object
LOCATION                object
POSTCODE                 int64
AREA_DESCRIPTION        object
REGION_DESCRIPTION      object
Unnamed: 10            float64
dtype: object

Null values per column:
PUBLISH_DATE               0
TRADING_NAME               0
BRAND_DESCRIPTION          0
PRODUCT_DESCRIPTION        0
PRODUCT_PRICE              0
ADDRESS                    0
LOCATION                   0
POSTCODE                   0
AREA_DESCRIPTION           0
REGION_DESCRIPTION         0
Unnamed: 10            82108
dtype: int64


In [12]:
# See what brands appear in the sample file
# The column might be called 'Brand', 'BRAND', or similar so adjust if needed
brand_col = [c for c in sample.columns if 'brand' in c.lower()][0]
print(f"Brand column name: '{brand_col}'")
print("\nAll brands in this file:")
print(sorted(sample[brand_col].unique()))

Brand column name: 'BRAND_DESCRIPTION'

All brands in this file:
['7-Eleven', 'Ampol', 'Atlas', 'BOC', 'BP', 'Better Choice', 'CGL Fuel', 'Caltex', 'Caltex Woolworths', 'Coles Express', 'Costco', 'Eagle', 'FastFuel 24/7', 'Gull', 'Independent', 'Liberty', 'Maisey Fuels', 'Metro Petroleum', 'Mobil', 'Puma', 'Shell', 'United', 'Vibe', 'WA Fuels']


In [13]:
# See what product/fuel types appear
product_col = [c for c in sample.columns if 'product' in c.lower() or 'fuel' in c.lower()][0]
print(f"Product column name: '{product_col}'")
print("\nAll product codes in this file:")
print(sorted(sample[product_col].unique()))

Product column name: 'PRODUCT_DESCRIPTION'

All product codes in this file:
['98 RON', 'Brand Diesel', 'Diesel', 'E85', 'LPG', 'PULP', 'ULP']


### Step 2 — Load and combine all files

Now we load all 40 monthly files and concatenate them into a single dataframe. We also standardise column names here so the rest of the notebooks use consistent, clean names regardless of any minor variations in the raw files.

In [15]:
# Load all files and combine
# We track which file each row came from to help with debugging if anything looks odd

frames = []
for filepath in raw_files:
    df = pd.read_csv(filepath)
    df['source_file'] = os.path.basename(filepath)
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)

print(f"Total rows loaded: {len(raw):,}")
print(f"Date range in raw data: {raw.iloc[:, 0].min()} to {raw.iloc[:, 0].max()}")
print(f"\nColumn names in combined data:")
print(raw.columns.tolist())

Total rows loaded: 3,384,550
Date range in raw data: 01/01/2022 to 31/12/2024

Column names in combined data:
['PUBLISH_DATE', 'TRADING_NAME', 'BRAND_DESCRIPTION', 'PRODUCT_DESCRIPTION', 'PRODUCT_PRICE', 'ADDRESS', 'LOCATION', 'POSTCODE', 'AREA_DESCRIPTION', 'REGION_DESCRIPTION', 'Unnamed: 10', 'source_file']


### Step 3 — Standardise column names

FuelWatch files have been known to use slightly different column names across years (e.g. `PRICE` vs `Price` vs `price`). The cell below maps whatever the raw columns are called to a clean, consistent set of names. 

**If this cell raises a KeyError**, it means one of the column names in the mapping below doesn't match what's in your files. Run `print(raw.columns.tolist())` and update the mapping accordingly.

In [17]:
# Normalise all column names to lowercase first to handle capitalisation differences
raw.columns = raw.columns.str.strip().str.lower().str.replace(' ', '_')

print("Normalised column names:")
print(raw.columns.tolist())

Normalised column names:
['publish_date', 'trading_name', 'brand_description', 'product_description', 'product_price', 'address', 'location', 'postcode', 'area_description', 'region_description', 'unnamed:_10', 'source_file']


In [18]:
# Map to our standard clean column names
# FuelWatch columns are typically: date, product, brand, price, suburb, region, 
# trading_name (station name), address, phone, latitude, longitude
#
# Adjust the keys below if your column names differ from what's shown

raw.columns = raw.columns.str.strip().str.lower().str.replace(' ', '_')

COLUMN_MAP = {
    'publish_date':       'date',
    'brand_description':  'brand',
    'product_description':'product_name',   # text like "Unleaded Petrol", not a numeric code
    'product_price':      'price_cpl',
    'trading_name':       'station_name',
    'address':            'address',
    'location':           'suburb',
    'area_description':   'region',
}

cols_present = {k: v for k, v in COLUMN_MAP.items() if k in raw.columns}
df = raw[list(cols_present.keys()) + ['source_file']].rename(columns=cols_present)

print("Columns after mapping:")
print(df.columns.tolist())
print(f"\nRows retained: {len(df):,}")
print(f"\nUnique product descriptions:")
print(sorted(df['product_name'].unique()))

Columns after mapping:
['date', 'brand', 'product_name', 'price_cpl', 'station_name', 'address', 'suburb', 'region', 'source_file']

Rows retained: 3,384,550

Unique product descriptions:
['98 RON', 'Brand Diesel', 'Diesel', 'E85', 'LPG', 'PULP', 'ULP']


### Step 4 — Parse dates and filter to unleaded petrol

FuelWatch product codes:
- **1** = Unleaded Petrol (ULP / E10 regular)
- **2** = Premium Unleaded (95 RON)
- **3** = Premium Unleaded (98 RON)
- **4** = Diesel
- **5** = LPG
- **6** = Brand Diesel
- **10** = E10

We focus on **Product 1 (ULP)** because it is the highest-volume fuel type and the one most relevant to 7-Eleven's retail pricing strategy. Diesel and premium are separate market segments with different competitive dynamics.

In [20]:
# Parse the date column
df['date'] = pd.to_datetime(df['date'], dayfirst=True)

print("Date range after parsing:")
print(f"  Earliest: {df['date'].min().date()}")
print(f"  Latest:   {df['date'].max().date()}")
print(f"  Unique dates: {df['date'].nunique():,}")

Date range after parsing:
  Earliest: 2021-12-01
  Latest:   2025-05-31
  Unique dates: 1,247


In [21]:
print(df.columns.tolist())

['date', 'brand', 'product_name', 'price_cpl', 'station_name', 'address', 'suburb', 'region', 'source_file']


In [22]:
print(raw.columns.tolist())

['publish_date', 'trading_name', 'brand_description', 'product_description', 'product_price', 'address', 'location', 'postcode', 'area_description', 'region_description', 'unnamed:_10', 'source_file']


In [23]:
# Filter to unleaded petrol by name — print unique values above first to confirm exact wording
ulp_keywords = ['unleaded', 'ulp', 'e10']
mask = df['product_name'].str.lower().str.contains('|'.join(ulp_keywords), na=False)
df_ulp = df[mask].copy()

print(f"\nProduct names retained after filter:")
print(df_ulp['product_name'].unique())
print(f"\nRows for unleaded petrol: {len(df_ulp):,}")
print(f"Rows dropped (other fuel types): {len(df) - len(df_ulp):,}")
print(f"\nPrice range in cents per litre:")
print(f"  Min: {df_ulp['price_cpl'].min()}")
print(f"  Max: {df_ulp['price_cpl'].max()}")
print(f"  Mean: {df_ulp['price_cpl'].mean():.1f}")


Product names retained after filter:
['ULP' 'PULP']

Rows for unleaded petrol: 1,454,125
Rows dropped (other fuel types): 1,930,425

Price range in cents per litre:
  Min: 125.9
  Max: 276.9
  Mean: 189.2


### Step 5 — Identify and handle outliers

Any price below 70 cpl or above 350 cpl is almost certainly a data entry error. Prices this far outside the plausible range do exist in the raw data — they are usually caused by a station entering 1650 instead of 165.0, or a test record that was never removed. We remove them rather than impute, because we cannot know what the correct price should have been.

In [25]:
# Inspect the price distribution before cleaning
print("Price distribution (cpl) — before cleaning:")
print(df_ulp['price_cpl'].describe().round(1))

# Flag obvious outliers
low_outliers  = df_ulp[df_ulp['price_cpl'] < 70]
high_outliers = df_ulp[df_ulp['price_cpl'] > 350]

print(f"\nPrices below 70 cpl: {len(low_outliers)} rows")
print(f"Prices above 350 cpl: {len(high_outliers)} rows")

if len(low_outliers) > 0:
    print("\nSample of low outliers:")
    print(low_outliers[['date', 'brand', 'suburb', 'price_cpl']].head(5))

if len(high_outliers) > 0:
    print("\nSample of high outliers:")
    print(high_outliers[['date', 'brand', 'suburb', 'price_cpl']].head(5))

Price distribution (cpl) — before cleaning:
count    1454125.0
mean         189.2
std           18.9
min          125.9
25%          175.9
50%          186.9
75%          199.9
max          276.9
Name: price_cpl, dtype: float64

Prices below 70 cpl: 0 rows
Prices above 350 cpl: 0 rows


In [26]:
# Remove outliers
df_clean = df_ulp[(df_ulp['price_cpl'] >= 70) & (df_ulp['price_cpl'] <= 350)].copy()

rows_removed = len(df_ulp) - len(df_clean)
print(f"Rows removed as outliers: {rows_removed} ({rows_removed/len(df_ulp)*100:.2f}% of ULP data)")
print(f"Rows remaining: {len(df_clean):,}")
print("\nPrice distribution after cleaning:")
print(df_clean['price_cpl'].describe().round(1))

Rows removed as outliers: 0 (0.00% of ULP data)
Rows remaining: 1,454,125

Price distribution after cleaning:
count    1454125.0
mean         189.2
std           18.9
min          125.9
25%          175.9
50%          186.9
75%          199.9
max          276.9
Name: price_cpl, dtype: float64


### Step 6 — Understand the brand landscape

Before filtering to our brands of interest, we look at the full list of brands in the data and their coverage. This tells us which brands have enough observations to be meaningful in a comparison analysis, and confirms that 7-Eleven is present and well-represented.

In [28]:
# Count observations and unique stations per brand
brand_summary = (
    df_clean
    .groupby('brand')
    .agg(
        total_observations=('price_cpl', 'count'),
        unique_stations=('station_name', 'nunique'),
        avg_price=('price_cpl', 'mean'),
        first_date=('date', 'min'),
        last_date=('date', 'max')
    )
    .sort_values('total_observations', ascending=False)
    .round(1)
)

print("All brands in dataset (sorted by number of observations):")
print(brand_summary.to_string())

All brands in dataset (sorted by number of observations):
                   total_observations  unique_stations  avg_price first_date  last_date
brand                                                                                  
BP                             263697              152      191.8 2021-12-01 2025-05-31
Ampol                          234864              122      191.2 2021-12-01 2025-05-31
Coles Express                  146556               81      194.2 2021-12-01 2025-05-13
7-Eleven                       128298               58      188.4 2021-12-01 2025-05-31
Caltex                         103225              130      190.1 2021-12-01 2025-05-31
United                          81359               42      184.8 2021-12-01 2025-05-31
Vibe                            75713               50      179.8 2021-12-01 2025-05-31
Puma                            73450               81      187.5 2021-12-01 2025-05-31
EG Ampol                        66404               40      19

In [29]:
# Confirm 7-Eleven is present
seven_eleven_brands = [b for b in df_clean['brand'].unique() if '7' in b or 'seven' in b.lower()]
print("7-Eleven related brand names in data:")
print(seven_eleven_brands)

7-Eleven related brand names in data:
['7-Eleven', 'FastFuel 24/7']


### Step 7 — Select competitor brands

We focus on the major national brands that 7-Eleven competes with directly. Smaller independent operators are excluded because they have inconsistent coverage and do not represent the systematic competitor pricing behaviour we are trying to analyse.

In [31]:
# Define the brands we want to keep
# Adjust this list based on what you see in the brand_summary output above
# Common FuelWatch brand names for the major players:

BRANDS_OF_INTEREST = [
    '7-Eleven',
    'Ampol',
    'BP',
    'Coles Express',  # now rebranded to Ampol in some locations — keep both
    'Shell',          # Viva Energy operates Shell-branded sites
    'United',
    'Puma',
    'Woolworths',     # Woolworths/Caltex sites in WA
    'Metro',
]

# Filter — only keep rows where brand matches our list
# We use a case-insensitive partial match to handle minor naming variations
pattern = '|'.join(BRANDS_OF_INTEREST)
df_filtered = df_clean[df_clean['brand'].str.contains(pattern, case=False, na=False)].copy()

print(f"Rows after brand filter: {len(df_filtered):,}")
print(f"\nBrands retained:")
print(sorted(df_filtered['brand'].unique()))

Rows after brand filter: 1,064,726

Brands retained:
['7-Eleven', 'Ampol', 'BP', 'Caltex Woolworths', 'Coles Express', 'EG Ampol', 'Metro Petroleum', 'Puma', 'Shell', 'United']


In [32]:
# Normalise brand names to a clean standard label
# This ensures 'Coles Express' and 'Coles Express (Woolworths)' etc. are treated as one brand
# Update this mapping based on what you find in your data

def normalise_brand(brand_name):
    b = brand_name.strip()
    if '7-eleven' in b.lower() or '7eleven' in b.lower():
        return '7-Eleven'
    elif 'ampol' in b.lower():
        return 'Ampol'
    elif 'bp' == b.upper() or b.upper().startswith('BP '):
        return 'BP'
    elif 'coles' in b.lower():
        return 'Coles Express'
    elif 'shell' in b.lower():
        return 'Shell'
    elif 'united' in b.lower():
        return 'United'
    elif 'puma' in b.lower():
        return 'Puma'
    elif 'woolworths' in b.lower():
        return 'Woolworths'
    elif 'metro' in b.lower():
        return 'Metro'
    else:
        return b

df_filtered['brand_clean'] = df_filtered['brand'].apply(normalise_brand)

print("Brand names after normalisation:")
print(df_filtered['brand_clean'].value_counts())

Brand names after normalisation:
brand_clean
Ampol            301268
BP               263697
Coles Express    146556
7-Eleven         128298
United            81359
Puma              73450
Shell             37988
Woolworths        24314
Metro              7796
Name: count, dtype: int64


### Step 8 — Final data quality checks

Before saving, we run a few checks to make sure the cleaned dataset is internally consistent and ready for analysis.

In [34]:
# Check 1: Are there any remaining nulls in the columns we care about?
key_cols = ['date', 'brand_clean', 'price_cpl', 'suburb', 'region']
print("Null counts in key columns:")
print(df_filtered[key_cols].isnull().sum())

Null counts in key columns:
date           0
brand_clean    0
price_cpl      0
suburb         0
region         0
dtype: int64


In [35]:
# Check 2: Do all brands have continuous coverage across the date range?
# Large gaps might indicate a brand entered or exited the WA market during our window

coverage = (
    df_filtered
    .groupby('brand_clean')['date']
    .agg(['min', 'max', 'nunique'])
    .rename(columns={'min': 'first_date', 'max': 'last_date', 'nunique': 'days_with_data'})
)

total_days = (df_filtered['date'].max() - df_filtered['date'].min()).days + 1
coverage['coverage_pct'] = (coverage['days_with_data'] / total_days * 100).round(1)

print(f"Total days in dataset: {total_days}")
print("\nCoverage by brand:")
print(coverage.to_string())

Total days in dataset: 1278

Coverage by brand:
              first_date  last_date  days_with_data  coverage_pct
brand_clean                                                      
7-Eleven      2021-12-01 2025-05-31            1247          97.6
Ampol         2021-12-01 2025-05-31            1247          97.6
BP            2021-12-01 2025-05-31            1247          97.6
Coles Express 2021-12-01 2025-05-13            1229          96.2
Metro         2021-12-01 2025-05-31            1247          97.6
Puma          2021-12-01 2025-05-31            1247          97.6
Shell         2021-12-01 2025-05-31            1247          97.6
United        2021-12-01 2025-05-31            1247          97.6
Woolworths    2021-12-01 2022-11-30             365          28.6


In [36]:
# Check 3: How many unique stations does 7-Eleven have in the data?
# This tells us the depth of our 7-Eleven sample

se_stations = df_filtered[df_filtered['brand_clean'] == '7-Eleven']['station_name'].nunique()
print(f"Unique 7-Eleven stations in WA dataset: {se_stations}")

print("\nUnique stations per brand:")
print(df_filtered.groupby('brand_clean')['station_name'].nunique().sort_values(ascending=False))

Unique 7-Eleven stations in WA dataset: 58

Unique stations per brand:
brand_clean
Ampol            162
BP               152
Coles Express     81
Puma              81
7-Eleven          58
United            42
Woolworths        39
Shell             37
Metro              8
Name: station_name, dtype: int64


In [37]:
# Check 4: Spot check — does 7-Eleven's average price look reasonable?
# Compare it against the overall market average

market_avg = df_filtered['price_cpl'].mean()
se_avg = df_filtered[df_filtered['brand_clean'] == '7-Eleven']['price_cpl'].mean()

print(f"Market average price (all brands): {market_avg:.1f} cpl")
print(f"7-Eleven average price:            {se_avg:.1f} cpl")
print(f"Difference: {se_avg - market_avg:+.1f} cpl")

Market average price (all brands): 190.5 cpl
7-Eleven average price:            188.4 cpl
Difference: -2.1 cpl


### Step 9 — Save the cleaned dataset

We save in Parquet format rather than CSV for two reasons. First, Parquet preserves data types — the `date` column stays as a proper datetime rather than a string. Second, Parquet files are typically 5–10x smaller than equivalent CSVs for this kind of data, and load significantly faster in Notebooks 2 and 3.

We also save a lightweight CSV version of the daily brand averages, which is easier to inspect manually and share with non-technical colleagues.

In [39]:
# Select and order the final columns
final_cols = ['date', 'brand_clean', 'price_cpl', 'suburb', 'region', 'station_name', 'address']
# Only include columns that exist after our cleaning steps
final_cols = [c for c in final_cols if c in df_filtered.columns]

df_final = df_filtered[final_cols].sort_values(['date', 'brand_clean']).reset_index(drop=True)

print(f"Final dataset shape: {df_final.shape}")
print(f"\nFinal column names: {df_final.columns.tolist()}")
print("\nSample rows:")
df_final.head(10)

Final dataset shape: (1064726, 7)

Final column names: ['date', 'brand_clean', 'price_cpl', 'suburb', 'region', 'station_name', 'address']

Sample rows:


,date,brand_clean,price_cpl,suburb,region,station_name,address
0,2021-12-01,7-Eleven,189.9,ASCOT,South of River,7-Eleven Ascot,194 Great Eastern Hwy
1,2021-12-01,7-Eleven,201.9,ASCOT,South of River,7-Eleven Ascot,194 Great Eastern Hwy
2,2021-12-01,7-Eleven,189.9,BALCATTA,North of River,7-Eleven Balcatta,174 Balcatta Rd
3,2021-12-01,7-Eleven,201.9,BALCATTA,North of River,7-Eleven Balcatta,174 Balcatta Rd
4,2021-12-01,7-Eleven,189.9,BALGA,North of River,7-Eleven Balga,102 Princess Rd
5,2021-12-01,7-Eleven,201.9,BALGA,North of River,7-Eleven Balga,102 Princess Rd
6,2021-12-01,7-Eleven,189.9,BANKSIA GROVE,North of River,7-Eleven Banksia Grove,1/300 Joseph Banks Bvd
7,2021-12-01,7-Eleven,201.9,BANKSIA GROVE,North of River,7-Eleven Banksia Grove,1/300 Joseph Banks Bvd
8,2021-12-01,7-Eleven,149.9,BASSENDEAN,North of River,7-Eleven Bassendean,302-318 Collier Rd
9,2021-12-01,7-Eleven,161.9,BASSENDEAN,North of River,7-Eleven Bassendean,302-318 Collier Rd


In [40]:
# Save the full cleaned station-level dataset as Parquet
parquet_path = 'data/processed/fuelwatch_clean.parquet'
df_final.to_parquet(parquet_path, index=False)
print(f"Saved station-level data to: {parquet_path}")

# Also create and save a daily brand-average dataset
# This is what Notebooks 2 and 3 will primarily work with
daily_avg = (
    df_final
    .groupby(['date', 'brand_clean'])
    .agg(
        avg_price=('price_cpl', 'mean'),
        median_price=('price_cpl', 'median'),
        station_count=('station_name', 'nunique'),
        min_price=('price_cpl', 'min'),
        max_price=('price_cpl', 'max')
    )
    .round(2)
    .reset_index()
)

daily_avg_path = 'data/processed/daily_brand_averages.parquet'
daily_avg.to_parquet(daily_avg_path, index=False)

# CSV version for easy inspection
daily_avg.to_csv('data/processed/daily_brand_averages.csv', index=False)

print(f"Saved daily brand averages to: {daily_avg_path}")
print(f"Also saved as CSV: data/processed/daily_brand_averages.csv")
print(f"\nDaily averages shape: {daily_avg.shape}")
print("\nSample:")
daily_avg.head(10)

Saved station-level data to: data/processed/fuelwatch_clean.parquet
Saved daily brand averages to: data/processed/daily_brand_averages.parquet
Also saved as CSV: data/processed/daily_brand_averages.csv

Daily averages shape: (10323, 7)

Sample:


,date,brand_clean,avg_price,median_price,station_count,min_price,max_price
0,2021-12-01,7-Eleven,191.09,189.9,42,149.9,201.9
1,2021-12-01,Ampol,184.91,185.9,83,149.9,198.9
2,2021-12-01,BP,185.10,184.9,137,158.9,207.9
3,2021-12-01,Coles Express,192.32,192.9,79,159.9,210.9
4,2021-12-01,Metro,167.15,166.4,2,159.9,175.9
5,2021-12-01,Puma,183.41,185.7,80,147.7,198.7
6,2021-12-01,Shell,169.93,168.9,25,145.5,198.5
7,2021-12-01,United,162.80,162.9,36,149.9,197.9
8,2021-12-01,Woolworths,185.40,187.9,39,149.5,203.9
9,2021-12-02,7-Eleven,181.52,182.9,42,145.9,198.9


### Summary

This notebook has:

1. Loaded all monthly FuelWatch CSV files and combined them into a single dataframe
2. Standardised column names to handle any inconsistencies across years
3. Filtered to unleaded petrol (Product 1) — the highest-volume and most competitively contested fuel type
4. Removed price outliers that indicate data entry errors
5. Filtered to major national retail brands and normalised their names
6. Validated data quality and coverage before saving
7. Saved two output files:
   - `fuelwatch_clean.parquet` — full station-level data, for detailed analysis
   - `daily_brand_averages.parquet` — daily average price per brand, for trend analysis

**Proceed to Notebook 2** to begin the competitor pricing analysis.

---
*Data source: FuelWatch Western Australia, Department of Energy, Mines, Industry Regulation and Safety.  
Licence: Creative Commons Attribution 4.0. Acknowledgement: FuelWatch (www.fuelwatch.wa.gov.au)*